In [2]:
import pandas as pd
import os
from tqdm.notebook import tqdm
# Change the 'basepath' to the directory of unzipped movie dataset
basepath = "C:\\Users\\HIMANSHU GUPTA\\$ML_PATH\\aclImdb_v1\\aclImdb"
labels = {'pos': 1, 'neg': 0}
data = []
for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        files = sorted(os.listdir(path))
        for file in tqdm(files, desc=f"Loading {s}/{l}"):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()
                data.append([txt, labels[l]])
df = pd.DataFrame(data, columns=['review', 'sentiment'])
print("\nDataset loaded successfully!")
print(df.shape)
print(df.head())

Loading test/pos:   0%|          | 0/12500 [00:00<?, ?it/s]

Loading test/neg:   0%|          | 0/12500 [00:00<?, ?it/s]

Loading train/pos:   0%|          | 0/12500 [00:00<?, ?it/s]

Loading train/neg:   0%|          | 0/12500 [00:00<?, ?it/s]


Dataset loaded successfully!
(50000, 2)
                                              review  sentiment
0  I went and saw this movie last night after bei...          1
1  Actor turned director Bill Paxton follows up h...          1
2  As a recreational golfer with some knowledge o...          1
3  I saw this film in a sneak preview, and it is ...          1
4  Bill Paxton has taken the true story of the 19...          1


In [1]:
# Storing the assembled and shuffled movie review dataset as a csv file
import numpy as np
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')

NameError: name 'df' is not defined

In [2]:
import pandas as pd
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df = df.rename(columns={"0": "review", "1": "sentiment"})
df.head(3)
                 

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


In [3]:
df.shape

(50000, 2)

# Bag-of-Word model

### Transforming words into feature vectors

In [4]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer()
docs = np.array(['The sun is shining', 'The weather is sweet', 'The sun is shining , the weather is sweet, and one and one is two'])
bag = count.fit_transform(docs)

In [5]:
print(count.vocabulary_)

{'the': 6, 'sun': 4, 'is': 1, 'shining': 3, 'weather': 8, 'sweet': 5, 'and': 0, 'one': 2, 'two': 7}


In [6]:
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]


### Assessing word relavancy via term frequency-inverse document frecuency

In [7]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf = TfidfTransformer(use_idf=True, norm='l2', smooth_idf=True)
np.set_printoptions(precision=2)
print(tfidf.fit_transform(count.fit_transform(docs)).toarray())

[[0.   0.43 0.   0.56 0.56 0.   0.43 0.   0.  ]
 [0.   0.43 0.   0.   0.   0.56 0.43 0.   0.56]
 [0.5  0.45 0.5  0.19 0.19 0.19 0.3  0.25 0.19]]


### Cleaning text data

In [8]:
df.loc[0, 'review'][-50:]

'is seven.<br /><br />Title (Brazil): Not Available'

In [ ]:
import re
def preprocessor(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = (re.sub(r'[\W]+',' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    return text

<>:4: SyntaxWarning: invalid escape sequence '\)'
<>:5: SyntaxWarning: invalid escape sequence '\W'
<>:4: SyntaxWarning: invalid escape sequence '\)'
<>:5: SyntaxWarning: invalid escape sequence '\W'
C:\Users\HIMANSHU GUPTA\AppData\Local\Temp\ipykernel_13752\2276463283.py:4: SyntaxWarning: invalid escape sequence '\)'
  emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
C:\Users\HIMANSHU GUPTA\AppData\Local\Temp\ipykernel_13752\2276463283.py:5: SyntaxWarning: invalid escape sequence '\W'
  text = (re.sub('[\W]+',' ', text.lower()) + ' '.join(emoticons).replace('-', ''))


In [10]:
preprocessor(df.loc[0, 'review'][-50:])

'is seven title brazil not available'

In [11]:
preprocessor("</a>This :) is :( a test :-)!")

'this is a test :) :( :)'

In [12]:
# Applying preprocessor function to all the movie reviews in our DataFrame
df['review'] = df['review'].apply(preprocessor)

### Processing documents into tokens

In [13]:
# one way to tokenize documents is to split them into individual words by splitting the cleaned documents at their whitespace characters
def tokenizer(text):
    return text.split()
tokenizer('runners like running and thus they run')

['runners', 'like', 'running', 'and', 'thus', 'they', 'run']

In [14]:
# Word stimming is the process of transforming word into its root form
from nltk.stem.porter import PorterStemmer
porter = PorterStemmer()
def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]
tokenizer_porter('runner like running thus they run')

['runner', 'like', 'run', 'thu', 'they', 'run']

In [15]:
# While stemming can create non-real words, such as 'thu'(from thus), as shown in the previous example,
# a technique called lemmatization aims to obtain the conical(grammatical correct) forms of the individual words-the so called lemmas

In [16]:
# 'Stop word removal': these are simply the words that are extremely common in all shorts of text and probably bear no (or only little)
# information that can be used to distinguish between different class of documents. Examples are is, and, has, and like.
import nltk
nltk.download('stopwords')

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [17]:
from nltk.corpus import stopwords
stop = stopwords.words('english')
[w for w in tokenizer_porter('a runner likes running and runs a lot') if w not in stop]

['runner', 'like', 'run', 'run', 'lot']

### Training a logistic regression model for document classification

In [18]:
# First we will divide dataframe of cleaned text documents into 25000 documents for training and 25000 documents for testing
X_train = df.loc[:25000, 'review'].values
y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
y_test = df.loc[25000:, 'sentiment'].values

In [19]:
# we will use GridSearchCV object to find optimal set of parameter for our logistic regression model using 5-fold stratified cross-validation
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(strip_accents=None, lowercase=False, preprocessor=None)
small_param_grid = [
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [None],
        'vect__tokenizer': [tokenizer, tokenizer_porter],
        'clf__l1_ratio': [0],
        'clf__C': [1.0, 10.0]
    },
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [stop, None],
        'vect__tokenizer': [tokenizer],
        'vect__use_idf': [False],
        'clf__l1_ratio': [0],
        'clf__C': [1.0, 10.0]
    }
]
lr_tfidf = Pipeline([
    ('vect', tfidf),
    ('clf', LogisticRegression(solver='liblinear'))
])
gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid, scoring='accuracy', cv=5, verbose=2, n_jobs=1)
gs_lr_tfidf.fit(X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


c:\Users\HIMANSHU GUPTA\$ML_PATH\my_env\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


[CV] END clf__C=1.0, clf__l1_ratio=0, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x000002E1D15D0360>; total time=   3.7s


c:\Users\HIMANSHU GUPTA\$ML_PATH\my_env\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


[CV] END clf__C=1.0, clf__l1_ratio=0, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x000002E1D15D0360>; total time=   4.0s


c:\Users\HIMANSHU GUPTA\$ML_PATH\my_env\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


... (73 outputs left out)

c:\Users\HIMANSHU GUPTA\$ML_PATH\my_env\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


[CV] END clf__C=10.0, clf__l1_ratio=0, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x000002E1D15D0360>, vect__use_idf=False; total time=   3.7s


c:\Users\HIMANSHU GUPTA\$ML_PATH\my_env\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'clf__C': [1.0, 10.0], 'clf__l1_ratio': [0], 'vect__ngram_range': [(1, ...)], 'vect__stop_words': [None], ...}, {'clf__C': [1.0, 10.0], 'clf__l1_ratio': [0], 'vect__ngram_range': [(1, ...)], 'vect__stop_words': [['a', 'about', ...], None], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"

In [20]:
print(f'Best parameter set: {gs_lr_tfidf.best_params_}')

Best parameter set: {'clf__C': 10.0, 'clf__l1_ratio': 0, 'vect__ngram_range': (1, 1), 'vect__stop_words': None, 'vect__tokenizer': <function tokenizer at 0x000002E1D15D0360>}


In [21]:
print(f'CV accuracy: {gs_lr_tfidf.best_score_:.3f}')
clf = gs_lr_tfidf.best_estimator_
print(f'Test Accuracy: {clf.score(X_test, y_test):.3f}')

CV accuracy: 0.897
Test Accuracy: 0.899


### Working with bigger data - online algorithms and out of-core learning

In [31]:
import numpy as np
import re
from nltk.corpus import stopwords
stop = stopwords.words('english')
def tokenizer(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:\)|\(|D|P)', text.lower())
    text = re.sub(r'[\W]+', ' ', text.lower()) \
        + ' '.join(emoticons).replace('-', '')
    tokenized = [w for w in text.split() if w not in stop]
    return tokenized

In [32]:
def stream_docs(path):
    with open(path, 'r', encoding='utf-8') as csv:
        next(csv) # skip header
        for line in csv:
            text, label = line[:-3], int(line[-2])
            yield text, label

In [33]:
next(stream_docs(path='movie_data.csv'))

('"In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70\'s, they discover the criminal and a net of power and money to cover the murder.<br /><br />""Murder in Greenwich"" is a good TV movie, with the true story of a murder of a fifteen years old girl that was committed by a wealthy teenager whose mother was a Kennedy. The powerful and rich f

In [46]:
def get_minibatch(doc_stream, size):
    docs, labels = [], []
    try:
        for _ in range(size):
            text, label = next(doc_stream)
            docs.append(text)
            labels.append(label)
    except StopIteration:
        pass
    if len(docs) ==0:
        return None, None
    return docs, np.array(labels)

In [47]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
vect = HashingVectorizer(decode_error='ignore', n_features=2**21, preprocessor=None, tokenizer=tokenizer)
clf = SGDClassifier(loss='log_loss', random_state=1)
doc_stream = stream_docs(path='movie_data.csv')

In [48]:
X_train, y_train = get_minibatch(doc_stream, size=1000)
print('Number of documents:', len(X_train))
print('Number of labels:', len(y_train))

Number of documents: 1000
Number of labels: 1000


In [49]:
import pyprind
pbar = pyprind.ProgBar(45)
classes = np.array([0, 1])
for _ in range(45):
    X_train, y_train = get_minibatch(doc_stream, size=1000)
    if  X_train is None:
        break
    X_train = vect.transform(X_train)
    clf.partial_fit(X_train, y_train, classes=classes)
    pbar.update()

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:27


In [51]:
X_test, y_test = get_minibatch(doc_stream, size=5000)
X_test = vect.transform(X_test)
print(f'Accuracy: {clf.score(X_test, y_test):.3f}')

Accuracy: 0.867


In [52]:
clf = clf.partial_fit(X_test, y_test)

# Topic modeling with latent Drichlet allocation

## Decomposing text document with LDA

In [55]:
import pandas as pd
df = pd.read_csv('movie_data.csv', encoding='utf-8')
# the following is necessary  on some computers:
df = df.rename(columns={"0": "review", "1": "sentiment"})


In [79]:
from sklearn.feature_extraction.text  import CountVectorizer
count = CountVectorizer(stop_words='english', max_df = .5, max_features=10000)
X = count.fit_transform(df['review'].values)

In [80]:
from sklearn.decomposition import LatentDirichletAllocation
lda = LatentDirichletAllocation(n_components=20, random_state=123, learning_method='batch')
X_topics = lda.fit_transform(X)

In [81]:
lda.components_.shape

(20, 10000)

In [83]:
n_top_words = 5
feature_names = count.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    print(f'Topic {(topic_idx + 1)}:')
    print(' '.join([feature_names[i] for i in topic.argsort() [:-n_top_words - 1:-1]]))

Topic 1:
just funny like comedy guy
Topic 2:
story characters films scenes plot
Topic 3:
book read like michael just
Topic 4:
people world point like man
Topic 5:
time just like seen movies
Topic 6:
young father life man girl
Topic 7:
life love story man woman
Topic 8:
bad like just acting good
Topic 9:
horror just like killer house
Topic 10:
dr king like man cage
Topic 11:
role good cast version robert
Topic 12:
story great performance best role
Topic 13:
years time war people series
Topic 14:
game series characters time like
Topic 15:
action good series like great
Topic 16:
like car good plot just
Topic 17:
like just story animation time
Topic 18:
sex dvd women version films
Topic 19:
best great dance tony musical
Topic 20:
just really like good think


In [85]:
horror = X_topics[:, 8].argsort()[::-1]
for iter_idx, movie_idx in enumerate(horror[:5]):
    print(f'\nHorror movie #{(iter_idx + 1)}:')
    print(df['review'][movie_idx][:300], '...')


Horror movie #1:
The monster will look very familiar to you. So will the rest of the film, if you've seen a half-dozen of these teenagers-trapped-in-the-woods movies. Okay, so they're not teenagers, this time, but they may as well be. Three couples decide it might be a good idea to check out a nearly-abandoned ghost ...

Horror movie #2:
The monster will look very familiar to you. So will the rest of the film, if you've seen a half-dozen of these teenagers-trapped-in-the-woods movies. Okay, so they're not teenagers, this time, but they may as well be. Three couples decide it might be a good idea to check out a nearly-abandoned ghost ...

Horror movie #3:
This is another Alien imitation and not a very good one at that.Replace outer space with the South African desert,throw in the same ingredients,a group of people stranded in an inhospitable landscape, have them hunted down by an alien creature and you have the same old story of a very ordinary film  ...

Horror movie #4:
More of a mys